In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, json, csv, shutil, subprocess, sys

PROJECT = "/content/drive/MyDrive/KYNU truncated isoform"
DATA = os.path.join(PROJECT, "data")
RUNS = os.path.join(PROJECT, "results", "colabfold")
LOGS = os.path.join(PROJECT, "results", "logs")
for d in (RUNS, LOGS):
    os.makedirs(d, exist_ok=True)

REQUIRED = ["constructs.fasta", "colabfold_input.csv", "pairings.tsv", "construct_provenance.tsv"]
missing = [f for f in REQUIRED if not os.path.exists(os.path.join(DATA, f))]
if missing:
    raise SystemExit("missing inputs in %s: %s" % (DATA, missing))

JOBS = list(csv.DictReader(open(os.path.join(DATA, "colabfold_input.csv"))))
if not JOBS:
    raise SystemExit("colabfold_input.csv holds no jobs")
for job in JOBS:
    if ":" not in job["sequence"]:
        raise SystemExit("job %s is not a two-chain job" % job["id"])
print("%d jobs" % len(JOBS))


In [ ]:
if shutil.which("colabfold_batch") is None:
    subprocess.run("pip -q install colabfold[alphafold]", shell=True, check=True)
if shutil.which("colabfold_batch") is None:
    raise SystemExit("colabfold_batch is not on PATH after installation")

version = subprocess.run(["colabfold_batch", "--version"], capture_output=True, text=True)
with open(os.path.join(LOGS, "environment.json"), "w") as fh:
    json.dump({"colabfold_batch": (version.stdout or version.stderr).strip(),
               "python": sys.version}, fh, indent=1)


In [ ]:
NUM_MODELS = 5
NUM_RECYCLE = 3

for job in JOBS:
    out = os.path.join(RUNS, job["id"])
    done = os.path.join(out, "COMPLETE")
    if os.path.exists(done):
        continue
    os.makedirs(out, exist_ok=True)
    query = os.path.join(out, "query.csv")
    with open(query, "w") as fh:
        fh.write("id,sequence\n%s,%s\n" % (job["id"], job["sequence"]))
    log = os.path.join(LOGS, job["id"] + ".log")
    with open(log, "w") as fh:
        result = subprocess.run(
            ["colabfold_batch", "--num-models", str(NUM_MODELS),
             "--num-recycle", str(NUM_RECYCLE), query, out],
            stdout=fh, stderr=subprocess.STDOUT)
    if result.returncode != 0:
        raise SystemExit("%s failed, see %s" % (job["id"], log))
    if not [f for f in os.listdir(out) if f.endswith(".pdb")]:
        raise SystemExit("%s produced no model, see %s" % (job["id"], log))
    open(done, "w").close()


In [ ]:
SCORE_KEYS = ("ptm", "iptm", "actifptm", "plddt")
rows = []
for job in JOBS:
    out = os.path.join(RUNS, job["id"])
    files = sorted(f for f in os.listdir(out) if f.endswith(".json") and "scores" in f)
    if not files:
        raise SystemExit("%s has no score json" % job["id"])
    for name in files:
        d = json.load(open(os.path.join(out, name)))
        row = {"job": job["id"], "model": name}
        for key in SCORE_KEYS:
            value = d.get(key)
            if isinstance(value, list):
                value = sum(value) / len(value)
            row[key] = value
        if row["iptm"] is None:
            raise SystemExit("%s model %s carries no ipTM" % (job["id"], name))
        rows.append(row)

target = os.path.join(PROJECT, "results", "complex_scores.tsv")
with open(target, "w") as fh:
    writer = csv.DictWriter(fh, fieldnames=["job", "model"] + list(SCORE_KEYS), delimiter="\t")
    writer.writeheader()
    writer.writerows(rows)

absent = sorted({r["job"] for r in rows if r.get("actifptm") is None})
with open(os.path.join(LOGS, "actifptm_coverage.json"), "w") as fh:
    json.dump({"jobs_without_actifptm": absent}, fh, indent=1)
print("wrote %d rows to %s" % (len(rows), target))


In [ ]:
IPSAE_SOURCE = None

if IPSAE_SOURCE is None:
    raise SystemExit("IPSAE_SOURCE is unset: confirm the authors' repository against bioRxiv 2025.02.10.637595")

clone = "/content/ipsae"
if not os.path.isdir(clone):
    subprocess.run(["git", "clone", "--depth", "1", IPSAE_SOURCE, clone], check=True)

script = os.path.join(clone, "ipsae.py")
if not os.path.exists(script):
    raise SystemExit("ipsae.py not found in %s" % clone)

ipsae_rows = []
for job in JOBS:
    out = os.path.join(RUNS, job["id"])
    pae = sorted(f for f in os.listdir(out) if f.endswith(".json") and "predicted_aligned_error" in f)
    models = sorted(f for f in os.listdir(out) if f.endswith(".pdb"))
    if not pae or not models:
        raise SystemExit("%s lacks a PAE json or a model for ipSAE" % job["id"])
    for pae_file, model in zip(pae, models):
        result = subprocess.run([sys.executable, script, os.path.join(out, pae_file),
                                 os.path.join(out, model), "10", "15"],
                                capture_output=True, text=True)
        if result.returncode != 0:
            raise SystemExit("ipSAE failed on %s %s: %s" % (job["id"], model, result.stderr[-400:]))
        ipsae_rows.append({"job": job["id"], "model": model, "stdout": result.stdout})

with open(os.path.join(PROJECT, "results", "ipsae_raw.json"), "w") as fh:
    json.dump(ipsae_rows, fh, indent=1)


In [ ]:
import collections

scores = collections.defaultdict(list)
for row in csv.DictReader(open(os.path.join(PROJECT, "results", "complex_scores.tsv")), delimiter="\t"):
    scores[row["job"]].append(float(row["iptm"]))

roles = {r["job"]: r["role"] for r in csv.DictReader(open(os.path.join(DATA, "pairings.tsv")), delimiter="\t")}
summary = []
for job, values in scores.items():
    summary.append({"job": job, "role": roles.get(job, ""), "n_models": len(values),
                    "iptm_best": max(values), "iptm_mean": sum(values) / len(values)})
summary.sort(key=lambda r: -r["iptm_best"])

target = os.path.join(PROJECT, "results", "complex_summary.tsv")
with open(target, "w") as fh:
    writer = csv.DictWriter(fh, fieldnames=["job", "role", "n_models", "iptm_best", "iptm_mean"], delimiter="\t")
    writer.writeheader()
    writer.writerows(summary)
print(open(target).read())
